Training a Diffusion model over MNIST

(a) Data preprocessing

In [2]:
import torch
import torch.nn as nn
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import os
import math
from tqdm import tqdm
import matplotlib.pyplot as plt


from typing import Dict, Tuple
import torch.nn.functional as F
from torchvision import models, transforms
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np




(b) define time schedule

In [22]:
def ddpm_schedules(beta1, beta2, T):
    """
    Returns pre-computed schedules for DDIM sampling, training process.
    """
    assert beta1 < beta2 < 1.0, "beta1 and beta2 must be in (0, 1)"

    beta_t = (beta2 - beta1) * torch.arange(0, T + 1, dtype=torch.float32) / T + beta1
    sqrt_beta_t = torch.sqrt(beta_t)
    alpha_t = 1 - beta_t
    log_alpha_t = torch.log(alpha_t)
    alphabar_t = torch.cumsum(log_alpha_t, dim=0).exp()

    sqrtab = torch.sqrt(alphabar_t)
    oneover_sqrta = 1 / torch.sqrt(alpha_t)

    sqrtmab = torch.sqrt(1 - alphabar_t)
    mab_over_sqrtmab_inv = (1 - alpha_t) / sqrtmab

    return {
        "alpha_t": alpha_t,  # \alpha_t
        "oneover_sqrta": oneover_sqrta,  # 1/\sqrt{\alpha_t}
        "sqrt_beta_t": sqrt_beta_t,  # \sqrt{\beta_t}
        "alphabar_t": alphabar_t,  # \bar{\alpha_t}
        "sqrtab": sqrtab,  # \sqrt{\bar{\alpha_t}}
        "sqrtmab": sqrtmab,  # \sqrt{1-\bar{\alpha_t}}
        "mab_over_sqrtmab": mab_over_sqrtmab_inv,  # (1-\alpha_t)/\sqrt{1-\bar{\alpha_t}}
    }


n_T = 400 # 500

schedule_dict = ddpm_schedules(1e-4, 0.02, n_T)


alphabar_t = schedule_dict["alphabar_t"].to(device)
betabar_t =  (1 - alphabar_t).to(device)



# Given a particular time idx i
# x_i = sqrt(alphabar_t[i]) x_0 + sqrt(betabar_t[i]) * eps
# which is equivalent to x_i = sqrt(alphabar_t[i]) x_0 + sqrt(1-alphabar_t[i]) * eps

print(len(alphabar_t))
print(alphabar_t)

401
tensor([0.9999, 0.9998, 0.9996, 0.9993, 0.9990, 0.9987, 0.9983, 0.9978, 0.9973,
        0.9968, 0.9962, 0.9955, 0.9948, 0.9941, 0.9933, 0.9925, 0.9916, 0.9906,
        0.9896, 0.9886, 0.9875, 0.9864, 0.9852, 0.9840, 0.9827, 0.9814, 0.9800,
        0.9786, 0.9772, 0.9756, 0.9741, 0.9725, 0.9708, 0.9692, 0.9674, 0.9656,
        0.9638, 0.9619, 0.9600, 0.9581, 0.9561, 0.9540, 0.9519, 0.9498, 0.9476,
        0.9454, 0.9432, 0.9409, 0.9385, 0.9361, 0.9337, 0.9312, 0.9287, 0.9262,
        0.9236, 0.9210, 0.9183, 0.9156, 0.9129, 0.9101, 0.9073, 0.9045, 0.9016,
        0.8987, 0.8957, 0.8928, 0.8897, 0.8867, 0.8836, 0.8805, 0.8773, 0.8741,
        0.8709, 0.8677, 0.8644, 0.8611, 0.8577, 0.8544, 0.8510, 0.8475, 0.8441,
        0.8406, 0.8371, 0.8335, 0.8300, 0.8264, 0.8228, 0.8191, 0.8154, 0.8117,
        0.8080, 0.8043, 0.8005, 0.7967, 0.7929, 0.7891, 0.7853, 0.7814, 0.7775,
        0.7736, 0.7697, 0.7657, 0.7618, 0.7578, 0.7538, 0.7498, 0.7458, 0.7417,
        0.7376, 0.7336, 0.7295, 0.72

(b) Define a neural architecture for the diffusion model

In [23]:


class ResidualConvBlock(nn.Module):
    def __init__(
        self, in_channels: int, out_channels: int, is_res: bool = False
    ) -> None:
        super().__init__()
        '''
        standard ResNet style convolutional block
        '''
        self.same_channels = in_channels==out_channels
        self.is_res = is_res
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.is_res:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            # this adds on correct residual in case channels have increased
            if self.same_channels:
                out = x + x2
            else:
                out = x1 + x2
            return out / 1.414
        else:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            return x2


class UnetDown(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UnetDown, self).__init__()
        '''
        process and downscale the image feature maps
        '''
        layers = [ResidualConvBlock(in_channels, out_channels), nn.MaxPool2d(2)]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class UnetUp(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UnetUp, self).__init__()
        '''
        process and upscale the image feature maps
        '''
        layers = [
            nn.ConvTranspose2d(in_channels, out_channels, 2, 2),
            ResidualConvBlock(out_channels, out_channels),
            ResidualConvBlock(out_channels, out_channels),
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x, skip):
        x = torch.cat((x, skip), 1)

        x = self.model(x)
        return x


class EmbedFC(nn.Module):
    def __init__(self, input_dim, emb_dim):
        super(EmbedFC, self).__init__()
        '''
        generic one layer FC NN for embedding things
        '''
        self.input_dim = input_dim
        layers = [
            nn.Linear(input_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(-1, self.input_dim)
        return self.model(x)


class ContextUnet(nn.Module):
    def __init__(self, in_channels, n_feat = 256):
        super(ContextUnet, self).__init__()

        self.in_channels = in_channels
        self.n_feat = n_feat
        self.n_classes = n_classes

        self.init_conv = ResidualConvBlock(in_channels, n_feat, is_res=True)

        self.down1 = UnetDown(n_feat, n_feat)
        self.down2 = UnetDown(n_feat, 2 * n_feat)

        self.to_vec = nn.Sequential(nn.AvgPool2d(7), nn.GELU())

        self.timeembed1 = EmbedFC(1, 2*n_feat)
        self.timeembed2 = EmbedFC(1, 1*n_feat)

        self.up0 = nn.Sequential(
            # nn.ConvTranspose2d(6 * n_feat, 2 * n_feat, 7, 7), # when concat temb and cemb end up w 6*n_feat
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, 7, 7), # otherwise just have 2*n_feat
            nn.GroupNorm(8, 2 * n_feat),
            nn.ReLU(),
        )

        self.up1 = UnetUp(4 * n_feat, n_feat)
        self.up2 = UnetUp(2 * n_feat, n_feat)
        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),
            nn.GroupNorm(8, n_feat),
            nn.ReLU(),
            nn.Conv2d(n_feat, self.in_channels, 3, 1, 1),
        )

    def forward(self, x, t):
        # x is (noisy) image,  t is timestep

        x = self.init_conv(x)
        down1 = self.down1(x)
        down2 = self.down2(down1)
        hiddenvec = self.to_vec(down2)


        # embed context, time step
        temb1 = self.timeembed1(t).view(-1, self.n_feat * 2, 1, 1)
        temb2 = self.timeembed2(t).view(-1, self.n_feat, 1, 1)

        # could concatenate the context embedding here instead of adaGN
        # hiddenvec = torch.cat((hiddenvec, temb1, cemb1), 1)

        up1 = self.up0(hiddenvec)
        # up2 = self.up1(up1, down2) # if want to avoid add and multiply embeddings

        up2 = self.up1(up1+temb1, down2)  # add and multiply embeddings
        up3 = self.up2(up2+temb2, down1)
        out = self.out(torch.cat((up3, x), 1))
        return out





(c) Train the diffusion model

In [ ]:

# hardcoding these here
n_epoch = 20
batch_size = 256
n_T = 400 # 500
device = "cuda:0"
n_classes = 10
n_feat = 128 # 128 ok, 256 better (but slower)
lrate = 1e-4

diffusion = ContextUnet(in_channels=1, n_feat=n_feat).to(device)

loss_mse = nn.MSELoss()

# optionally load a model
# ddpm.load_state_dict(torch.load("./data/diffusion_outputs/ddpm_unet01_mnist_9.pth"))

tf = transforms.Compose([transforms.ToTensor()]) # mnist is already normalised 0 to 1

dataset = MNIST("./data", train=True, download=True, transform=tf)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=5)

optim = torch.optim.Adam(diffusion.parameters(), lr=lrate)

for ep in range(n_epoch):
  print(f'epoch {ep}')
  diffusion.train()

  # linear lrate decay
  optim.param_groups[0]['lr'] = lrate*(1-ep/n_epoch)

  pbar = tqdm(dataloader)
  loss_ema = None
  for x_0, _ in pbar:
      optim.zero_grad()
      x_0 = x_0.to(device)

      _ts = torch.randint(1, n_T+1, (x_0.shape[0],)).to(device)  # t ~ Uniform(0, n_T)
      noise = torch.randn_like(x_0)  # eps ~ N(0, 1)


      # We should predict the "error term" from x_t below.
      x_t = torch.sqrt(alphabar_t[_ts, None, None, None])*x_0 + torch.sqrt(betabar_t[_ts, None, None, None])*noise

      noise_predict = diffusion(x_t, _ts / n_T)

      loss = loss_mse(noise_predict, noise)

      loss.backward()
      optim.step()

  # for eval, save an image of currently generated samples (top rows)
  # followed by real images (bottom rows)
  diffusion.eval()
  with torch.no_grad():
      n_sample = 4*n_classes

      x_i = torch.randn(n_sample, *(1, 28, 28)).to(device)  # x_T ~ N(0, 1), sample initial noise
      for i in range(n_T, 0, -1):
        t_is = torch.tensor([i / n_T]).to(device)

        eps = diffusion(x_i, t_is)
        x_0_pred = (x_i - torch.sqrt(1-alphabar_t[i])*eps)/torch.sqrt(alphabar_t[i])

        x_i = torch.sqrt(alphabar_t[i-1])*x_0_pred + torch.sqrt(1-alphabar_t[i-1])*eps


      samples = x_i.to('cpu').numpy()
      fig, axes = plt.subplots(1,  1, figsize=(5, 5), sharex=True, sharey=True)

      plt.imshow(samples[0,:,:,:].reshape(28,28), cmap=plt.get_cmap('gray'))
      plt.axis('off')  # Turn off axis labels
      plt.show()

